In [ ]:
#im using from https://www.kaggle.com/code/xiaoleilian/biohub-m001-ens3-sm6-sim2/notebook

import os, json, glob, time, gc
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max


DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        _p = nn.Conv3d(1,1,3).to("cuda")
        _ = _p(torch.zeros(1,1,4,4,4,device="cuda")).cpu()
        DEVICE = "cuda"
        del _p
    except Exception as e:
        print("GPU present but conv3d unusable -> CPU:", str(e)[:80])

# ----------------------------------------------------------------------
# Constants
# ----------------------------------------------------------------------
SCALE = np.array([1.625, 0.40625, 0.40625])
POOL = 4

WEIGHT_NAMES = ['unet3d_bright.pt', 'unet3d_traintophat.pt']
PREPROCS = ['', 'tophat']          # one per model; each uses its trained preproc

REPAIR = True
CAND_THR = 0.05
GAP_DT = 0                         # base gap closing disabled (patch handles 1f)
GAP_GATE_UM = 10.0
SNAP_UM = 3.0
SHORT_MIN = 6                      # overridden to 6 in final config
LINEFIT_WEIGHT = 0.8
LINEFIT_WINDOW = 2

UNET_THRESH = 0.15
NMS_UM = 4.0
DETECT_THRESH = min(UNET_THRESH, CAND_THR) if REPAIR else UNET_THRESH

MAX_LINK_UM = 10.0
TIGHT_UM = 6.0

# Appearance cost weight
LINK_SIM_W = 2.0

# ----------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------
CANDIROOT = [
    "/kaggle/input/biohub-cell-tracking-during-development",
    "/kaggle/input/competitions/biohub-cell-tracking-during-development",
    "data"
]
ROOT = next((p for p in CANDIROOT if Path(p, "test").exists()), "data")
TEST_DIR = Path(ROOT) / "test"
OUT = "submission.csv"

def _find_weight(name):
    cands = [
        f"/kaggle/input/biohub-unet3d-weights/{name}",
        f"models/{name}",
    ] + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    p = next((c for c in cands if Path(c).exists()), None)
    if p is None:
        raise FileNotFoundError(f"{name} not found; attach biohub-unet3d-weights.")
    return p

WEIGHTS = [_find_weight(n) for n in WEIGHT_NAMES]

print("device:", DEVICE, "| torch", torch.__version__)
print("models:", list(zip(WEIGHT_NAMES, [p or "none" for p in PREPROCS])))
print("repair:", REPAIR, "| seed_thr:", UNET_THRESH, "| short:", SHORT_MIN)
print("data:", ROOT, "| weights:", WEIGHTS)

# ----------------------------------------------------------------------
# UNet3D model definition
# ----------------------------------------------------------------------
def _block(ci, co):
    return nn.Sequential(
        nn.Conv3d(ci, co, 3, padding=1),
        nn.BatchNorm3d(co),
        nn.ReLU(inplace=True),
        nn.Conv3d(co, co, 3, padding=1),
        nn.BatchNorm3d(co),
        nn.ReLU(inplace=True),
    )

class UNet3D(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.e1 = _block(1, base)
        self.e2 = _block(base, base*2)
        self.e3 = _block(base*2, base*4)
        self.pool = nn.MaxPool3d(2)
        self.bott = _block(base*4, base*8)
        self.u3 = nn.ConvTranspose3d(base*8, base*4, 2, stride=2)
        self.d3 = _block(base*8, base*4)
        self.u2 = nn.ConvTranspose3d(base*4, base*2, 2, stride=2)
        self.d2 = _block(base*4, base*2)
        self.u1 = nn.ConvTranspose3d(base*2, base, 2, stride=2)
        self.d1 = _block(base*2, base)
        self.out = nn.Conv3d(base, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

# ----------------------------------------------------------------------
# Load base models (bright + tophat)
# ----------------------------------------------------------------------
MODELS = []
for _w in WEIGHTS:
    _ck = torch.load(_w, map_location=DEVICE)
    _m = UNet3D(base=_ck.get("base", 24)).to(DEVICE)
    _m.load_state_dict(_ck["state_dict"])
    _m.eval()
    MODELS.append(_m)
    print("loaded", Path(_w).name, "| val_recall", _ck.get("val_recall"), "| aug", _ck.get("aug"))

assert len(MODELS) == len(PREPROCS)

# ----------------------------------------------------------------------
# Load 3rd model: unet3d_v2_tophat_b32.pt
# ----------------------------------------------------------------------
_ck3 = torch.load(_find_weight('unet3d_v2_tophat_b32.pt'), map_location=DEVICE)
_m3 = UNet3D(base=_ck3.get('base', 24)).to(DEVICE)
_m3.load_state_dict(_ck3['state_dict'])
_m3.eval()
print("loaded", 'unet3d_v2_tophat_b32.pt', '| val_recall', _ck3.get('val_recall'))

# ----------------------------------------------------------------------
# Branch definitions for ensemble
# ----------------------------------------------------------------------
_BRANCHES = []
_BRANCHES.append(([MODELS[0]], [1.0], ''))               # bright, no preproc
_BRANCHES.append(([MODELS[1]], [1.0], 'tophat'))         # tophat v1
_BRANCHES.append(([_m3], [1.0], 'tophat'))               # tophat v2 (b32)

_FLIP_VIEWS = (0, 1, 2, 3)   # bit0=flipY, bit1=flipX

# ----------------------------------------------------------------------
# Utility: read zarr / blosc2 volume
# ----------------------------------------------------------------------
_ZC = {}

def read_array_meta(zp):
    with open(Path(zp) / "0" / "zarr.json") as f:
        m = json.load(f)
    return dict(shape=tuple(m["shape"]), dtype=np.dtype(m["data_type"]))

def load_volume(zp, t, meta=None):
    try:
        import zarr
        k = str(zp)
        if k not in _ZC:
            _ZC[k] = zarr.open(k, mode="r")["0"]
        return np.asarray(_ZC[k][t])
    except Exception:
        import blosc2
        if meta is None:
            meta = read_array_meta(zp)
        buf = blosc2.decompress(open(Path(zp) / "0" / "c" / str(t) / "0" / "0" / "0", "rb").read())
        return np.frombuffer(buf, dtype=meta["dtype"]).reshape(meta["shape"][1:])

# ----------------------------------------------------------------------
# Pooling, normalization, tophat
# ----------------------------------------------------------------------
def pool_xy(vol, f=POOL):
    Z, Y, X = vol.shape
    Y2, X2 = (Y // f) * f, (X // f) * f
    v = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return v.reshape(Z, Y2//f, f, X2//f, f).mean(axis=(2, 4))

def pool_norm(vol, preproc=""):
    p = pool_xy(vol)
    if preproc == "tophat":
        from scipy.ndimage import grey_opening
        p = np.clip(p - grey_opening(p, size=(1, 7, 7)), 0.0, None)
    lo = float(np.percentile(p, 50))
    hi = float(np.percentile(p, 99.5))
    return np.clip((p - lo) / (hi - lo + 1e-6), -0.5, 6.0).astype(np.float32)

# ----------------------------------------------------------------------
# Flip helper
# ----------------------------------------------------------------------
def _flip(x, i):
    if i & 1:
        x = np.flip(x, -2)
    if i & 2:
        x = np.flip(x, -1)
    return x

# ----------------------------------------------------------------------
# Refinement and physical NMS
# ----------------------------------------------------------------------
def _refine(vol, zyx, rz=2, ryx=5):
    Z, Y, X = vol.shape
    z, y, x = (int(round(v)) for v in zyx)
    z0, z1 = max(0, z-rz), min(Z, z+rz+1)
    y0, y1 = max(0, y-ryx), min(Y, y+ryx+1)
    x0, x1 = max(0, x-ryx), min(X, x+ryx+1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32)
    bg = float(crop.min())
    w = np.clip(crop - bg, 0, None)
    s = float(w.sum())
    if s <= 0:
        return np.array([z, y, x], float), 0.0
    zz, yy, xx = np.mgrid[z0:z1, y0:y1, x0:x1]
    return np.array([(zz*w).sum(), (yy*w).sum(), (xx*w).sum()]) / s, float(crop.max() - bg)

def _physical_nms(coords, scores, radius_um, scale=SCALE):
    if len(coords) <= 1:
        return coords, scores
    pts = coords * scale[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    killed = np.zeros(len(coords), bool)
    keep = []
    for i in order:
        if killed[i]:
            continue
        keep.append(int(i))
        killed[tree.query_ball_point(pts[i], r=radius_um)] = True
    keep = np.array(keep)
    return coords[keep], scores[keep]

# ----------------------------------------------------------------------
# Detection with 3-model ensemble + flip-quartet TTA
# ----------------------------------------------------------------------
def detect(vol):
    hs = []
    for models, ws, pp in _BRANCHES:
        x = pool_norm(vol, pp)
        acc = None
        for i in _FLIP_VIEWS:
            xv = np.ascontiguousarray(_flip(x, i))
            with torch.no_grad():
                lg = None
                for m, w in zip(models, ws):
                    l = m(torch.from_numpy(xv)[None, None].to(DEVICE))[0, 0].float().cpu().numpy()
                    lg = w * l if lg is None else lg + w * l
            lg = _flip(lg, i)   # inverse flip
            acc = lg if acc is None else acc + lg
        hs.append(1.0 / (1.0 + np.exp(-(acc / len(_FLIP_VIEWS)))))

    h = hs[0] if len(hs) == 1 else np.mean(hs, axis=0)

    pk = peak_local_max(h, min_distance=1, threshold_abs=DETECT_THRESH, exclude_border=False)
    if len(pk) == 0:
        return np.zeros((0, 3)), np.zeros(0)

    sc = h[pk[:, 0], pk[:, 1], pk[:, 2]].astype(float)
    coords = pk.astype(float)
    coords[:, 1] = coords[:, 1] * POOL + (POOL - 1) / 2
    coords[:, 2] = coords[:, 2] * POOL + (POOL - 1) / 2
    ref = np.array([_refine(vol, c)[0] for c in coords])
    return _physical_nms(ref, sc, NMS_UM)

# ----------------------------------------------------------------------
# Distance
# ----------------------------------------------------------------------
def _dist_um(a, b):
    d = (np.asarray(a, float) - np.asarray(b, float)) * SCALE
    return float(np.sqrt((d * d).sum()))

# ----------------------------------------------------------------------
# Logit for appearance cost
# ----------------------------------------------------------------------
def _logit(s):
    s = np.clip(s, 1e-4, 1 - 1e-4)
    return np.log(s / (1 - s))

# ----------------------------------------------------------------------
# Linker with appearance cost
# ----------------------------------------------------------------------
def _link_sim(prev_xyz, curr_xyz, prev_vel, prev_sc, curr_sc):
    if len(prev_xyz) == 0 or len(curr_xyz) == 0:
        return []
    P = prev_xyz * SCALE[None, :]
    C = curr_xyz * SCALE[None, :]
    pred = P + (0.5 * prev_vel if prev_vel is not None else 0.0)
    N, M = len(P), len(C)
    BIG = 1e9
    sim = LINK_SIM_W * np.abs(_logit(prev_sc)[:, None] - _logit(curr_sc)[None])

    def _hun(pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0:
            return []
        Draw = np.sqrt(((P[pi][:, None] - C[ci][None])**2).sum(2))
        D = np.sqrt(((pred[pi][:, None] - C[ci][None])**2).sum(2))
        cost = np.where(Draw > gate, BIG, D) + np.where(Draw > gate, 0.0, sim[np.ix_(pi, ci)])
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]

    links = _hun(np.arange(N), np.arange(M), min(TIGHT_UM, MAX_LINK_UM))
    up = {p for p, _ in links}
    uc = {c for _, c in links}
    fp = np.array([i for i in range(N) if i not in up], int)
    fc = np.array([j for j in range(M) if j not in uc], int)
    return links + _hun(fp, fc, MAX_LINK_UM)

# ----------------------------------------------------------------------
# Short filter (SHORT_MIN=6)
# ----------------------------------------------------------------------
def _short_filter(nodes, edges):
    if SHORT_MIN <= 1 or not edges:
        return nodes, edges
    parent = {nid: nid for nid in nodes}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        if a not in parent or b not in parent:
            return
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    out_count = defaultdict(int)
    for a, b in edges:
        union(a, b)
        out_count[a] += 1
    comps = defaultdict(list)
    for nid in nodes:
        comps[find(nid)].append(nid)
    keep = set()
    for members in comps.values():
        has_div = any(out_count[n] >= 2 for n in members)
        if len(members) >= SHORT_MIN or has_div:
            keep.update(members)
    nodes2 = {nid: n for nid, n in nodes.items() if nid in keep}
    edges2 = [(a, b) for a, b in edges if a in nodes2 and b in nodes2]
    return nodes2, edges2

# ----------------------------------------------------------------------
# Linefit smoothing
# ----------------------------------------------------------------------
def _linefit(nodes, edges):
    if LINEFIT_WEIGHT <= 0:
        return
    pred = defaultdict(list)
    succ = defaultdict(list)
    for a, b in edges:
        if a in nodes and b in nodes and int(nodes[b]["t"]) == int(nodes[a]["t"]) + 1:
            succ[a].append(b)
            pred[b].append(a)
    orig = {k: v["xyz"].copy() for k, v in nodes.items()}
    updates = {}
    W = int(LINEFIT_WINDOW)
    for nid in nodes:
        neigh = [(0, nid)]
        cur = nid
        for step in range(1, W + 1):
            ps = pred.get(cur, [])
            if len(ps) != 1:
                break
            cur = ps[0]
            neigh.append((-step, cur))
        cur = nid
        for step in range(1, W + 1):
            ss = succ.get(cur, [])
            if len(ss) != 1:
                break
            cur = ss[0]
            neigh.append((step, cur))
        if len(neigh) < 3:
            continue
        dt = np.array([a for a, _ in neigh], float)
        xyz = np.stack([orig[n] for _, n in neigh])
        fit = np.array([np.polyval(np.polyfit(dt, xyz[:, ax], 1), 0.0) for ax in range(3)])
        if np.isfinite(fit).all():
            updates[nid] = (1.0 - LINEFIT_WEIGHT) * orig[nid] + LINEFIT_WEIGHT * fit
    for nid, xyz in updates.items():
        nodes[nid]["xyz"] = xyz

# ----------------------------------------------------------------------
# Emit CSV rows
# ----------------------------------------------------------------------
COLS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]

def _emit(ds, nodes, edges):
    edge_set = []
    seen = set()
    for a, b in edges:
        if a == b or a not in nodes or b not in nodes or (a, b) in seen:
            continue
        seen.add((a, b))
        edge_set.append((a, b))
    used = set()
    for a, b in edge_set:
        used.add(a)
        used.add(b)
    nrows = []
    erows = []
    for nid in sorted(used):
        n = nodes[nid]
        z, y, x = n["xyz"]
        nrows.append((ds, "node", int(nid), int(n["t"]), float(z), float(y), float(x), -1, -1))
    for a, b in edge_set:
        if a in used and b in used:
            erows.append((ds, "edge", -1, -1, -1, -1, -1, int(a), int(b)))
    return pd.DataFrame(nrows, columns=COLS), pd.DataFrame(erows, columns=COLS)

# ----------------------------------------------------------------------
# PATCH 1: Safe divisions
# ----------------------------------------------------------------------
DIV_PARENT_UM = 12.0
DIV_SISTER_UM = 15.0
DIV_CHILD_UM = 10.0
DIV_DIVERGE_UM = 2.25
DIV_W_SISTER = 0.15
DIV_FRAME_CAP = 0.0076
DIV_GLOBAL_CAP = 0.00375

def _add_safe_divisions(nodes, edges):
    succ = defaultdict(list)
    pred = defaultdict(list)
    for a, b in edges:
        succ[a].append(b)
        pred[b].append(a)

    by_t = defaultdict(list)
    for nid, n in nodes.items():
        by_t[n["t"]].append(nid)

    orphan_tree = {}
    orphan_ids = {}
    for t, ids in by_t.items():
        orph = [g for g in ids if not pred.get(g)]
        orphan_ids[t] = orph
        if orph:
            orphan_tree[t] = cKDTree(np.asarray([nodes[g]["xyz"] for g in orph], float) * SCALE)

    proposals = []
    for p, n in nodes.items():
        ch = succ.get(p, [])
        if len(ch) != 1:
            continue
        c1 = ch[0]
        t = n["t"]
        if nodes[c1]["t"] != t + 1:
            continue
        if len(pred.get(p, [])) != 1:
            continue
        if _dist_um(n["xyz"], nodes[c1]["xyz"]) > DIV_CHILD_UM:
            continue
        tree = orphan_tree.get(t + 1)
        if tree is None:
            continue
        for idx in tree.query_ball_point(n["xyz"] * SCALE, r=DIV_PARENT_UM):
            q = orphan_ids[t + 1][idx]
            if q == c1:
                continue
            d_pq = _dist_um(n["xyz"], nodes[q]["xyz"])
            d_s = _dist_um(nodes[c1]["xyz"], nodes[q]["xyz"])
            if d_s > DIV_SISTER_UM:
                continue
            d_q, q_idx = tree.query(nodes[c1]["xyz"] * SCALE)
            if orphan_ids[t + 1][q_idx] != q or d_q > DIV_SISTER_UM:
                continue
            c1n, qn = succ.get(c1, []), succ.get(q, [])
            if len(c1n) != 1 or len(qn) != 1:
                continue
            if _dist_um(nodes[c1n[0]]["xyz"], nodes[qn[0]]["xyz"]) - d_s < DIV_DIVERGE_UM:
                continue
            proposals.append((d_pq + DIV_W_SISTER * d_s, t, p, q))

    proposals.sort()
    used_p = set()
    used_q = set()
    per_frame = defaultdict(int)
    n_frame = {t: len(ids) for t, ids in by_t.items()}
    max_global = max(1, int(DIV_GLOBAL_CAP * len(nodes)))
    added = 0

    for score, t, p, q in proposals:
        if added >= max_global:
            break
        if p in used_p or q in used_q:
            continue
        if per_frame[t] >= max(1, int(DIV_FRAME_CAP * n_frame[t])):
            continue
        edges.append((p, q))
        succ[p].append(q)
        pred[q].append(p)
        used_p.add(p)
        used_q.add(q)
        per_frame[t] += 1
        added += 1
    return added

# ----------------------------------------------------------------------
# PATCH 2: 1-frame gap closing (snap-only)
# ----------------------------------------------------------------------
GAP1_GATE_UM = 9.0
GAP1_SNAP_UM = 3.2
GAP1_MIN_CAND_SCORE = 0.10
GAP1_CAP_FRAC = 0.003

def _gap_close_1f_snap(nodes, edges, cand, cand_sc, cand_trees, next_id):
    T = len(cand)
    succ = {}
    pred = {}
    for a, b in edges:
        succ[a] = b
        pred[b] = a

    by_t = defaultdict(list)
    for nid, n in nodes.items():
        by_t[n["t"]].append(nid)

    cand_used = [np.zeros(len(c), bool) for c in cand]
    max_gaps = max(1, int(GAP1_CAP_FRAC * len(nodes)))
    n_added = 0
    BIG = 1e9

    for t in range(T - 2):
        if n_added >= max_gaps:
            break
        ends = [g for g in by_t.get(t, []) if g not in succ and g in pred]
        starts = [g for g in by_t.get(t + 2, []) if g not in pred and g in succ]
        if not ends or not starts:
            continue
        P = np.asarray([nodes[g]["xyz"] for g in ends], float) * SCALE
        C = np.asarray([nodes[g]["xyz"] for g in starts], float) * SCALE
        D = np.sqrt(((P[:, None] - C[None])**2).sum(2))
        cost = np.where(D > GAP1_GATE_UM, BIG, D)
        ri, ci = linear_sum_assignment(cost)
        props = sorted((float(D[r, c]), ends[r], starts[c]) for r, c in zip(ri, ci) if cost[r, c] < BIG)

        for d, ge, gs in props:
            if n_added >= max_gaps:
                break
            if ge in succ or gs in pred:
                continue
            mid = 0.5 * (nodes[ge]["xyz"] + nodes[gs]["xyz"])
            use = None
            tree = cand_trees[t + 1]
            if tree is not None:
                dist, idx = tree.query(mid * SCALE)
                if (dist <= GAP1_SNAP_UM and not cand_used[t + 1][idx] and
                        cand_sc[t + 1][idx] >= GAP1_MIN_CAND_SCORE):
                    use = np.asarray(cand[t + 1][idx], float)
                    cand_used[t + 1][idx] = True
            if use is None:
                continue
            ng = next_id
            next_id += 1
            nodes[ng] = {"t": t + 1, "xyz": use}
            edges.append((ge, ng))
            edges.append((ng, gs))
            succ[ge] = ng
            pred[ng] = ge
            succ[ng] = gs
            pred[gs] = ng
            n_added += 1
    return next_id, n_added

# ----------------------------------------------------------------------
# Main tracking pipeline (repair_track with patches)
# ----------------------------------------------------------------------
def repair_track(dets, ds):
    nodes = {}
    frame_ids = []
    cand = []
    cand_sc = []
    cand_trees = []
    nid = 1
    NSC = {}

    for t, (coords, scores) in enumerate(dets):
        coords = np.asarray(coords, float).reshape(-1, 3)
        scores = np.asarray(scores, float).reshape(-1)
        seeds = coords[scores >= UNET_THRESH]
        cm = (scores >= CAND_THR) & (scores < UNET_THRESH)
        cand.append(coords[cm])
        cand_sc.append(scores[cm])
        cand_trees.append(cKDTree(cand[-1] * SCALE) if len(cand[-1]) else None)
        seed_sc = scores[scores >= UNET_THRESH]
        ids = []
        for xyz, s_ in zip(seeds, seed_sc):
            nodes[nid] = {"t": t, "xyz": np.asarray(xyz, float)}
            NSC[nid] = float(s_)
            ids.append(nid)
            nid += 1
        frame_ids.append(ids)

    edges = []
    succ = {}
    pred = {}
    vel = {}

    for t in range(len(dets) - 1):
        P = np.asarray([nodes[g]["xyz"] for g in frame_ids[t]], float).reshape(-1, 3)
        C = np.asarray([nodes[g]["xyz"] for g in frame_ids[t + 1]], float).reshape(-1, 3)
        if len(P) == 0 or len(C) == 0:
            continue
        prev_vel = np.array([vel.get(g, np.zeros(3)) for g in frame_ids[t]])
        psc = np.array([NSC[g] for g in frame_ids[t]])
        csc = np.array([NSC[g] for g in frame_ids[t + 1]])
        for pi, ci in _link_sim(P, C, prev_vel if len(prev_vel) else None, psc, csc):
            gp, gc = frame_ids[t][pi], frame_ids[t + 1][ci]
            edges.append((gp, gc))
            succ[gp] = gc
            pred[gc] = gp
            vel[gc] = (C[ci] - P[pi]) * SCALE

    # PATCH 2: 1-frame gap closing (before short-filter)
    nid, n_gaps = _gap_close_1f_snap(nodes, edges, cand, cand_sc, cand_trees, nid)

    nodes, edges = _short_filter(nodes, edges)
    _linefit(nodes, edges)

    # PATCH 1: safe divisions (after linefit)
    n_div = _add_safe_divisions(nodes, edges)

    print(f"    patches[{ds}]: gaps={n_gaps} divs={n_div}")
    return _emit(ds, nodes, edges)

# ----------------------------------------------------------------------
# Track one movie
# ----------------------------------------------------------------------
def track_movie(zp, ds, T):
    if REPAIR:
        meta = read_array_meta(zp)
        dets = []
        for t in range(T):
            dets.append(detect(load_volume(zp, t, meta)))
            gc.collect()
        return repair_track(dets, ds)

    # Fallback: no repair (not used in final config)
    meta = read_array_meta(zp)
    node_rows = []
    edge_rows = []
    prev_ids = []
    prev_xyz = np.zeros((0, 3))
    prev_vel = None
    nid = 1
    for t in range(T):
        coords, scores = detect(load_volume(zp, t, meta))
        gc.collect()
        ids = list(range(nid, nid + len(coords)))
        nid += len(coords)
        for i, c in zip(ids, coords):
            node_rows.append((ds, "node", i, t, float(c[0]), float(c[1]), float(c[2]), -1, -1))
        if t > 0 and len(prev_ids):
            links = _link_sim(prev_xyz, coords, prev_vel, np.ones(len(prev_xyz)), np.ones(len(coords)))
            vel = np.zeros((len(prev_xyz), 3))
            for p, c in links:
                edge_rows.append((ds, "edge", -1, -1, -1, -1, -1, prev_ids[p], ids[c]))
                vel[p] = (coords[c] - prev_xyz[p]) * SCALE
            nv = np.zeros((len(coords), 3))
            for p, c in links:
                nv[c] = vel[p]
            prev_vel = nv
        else:
            prev_vel = None
        prev_ids, prev_xyz = ids, coords
    nodes = pd.DataFrame(node_rows, columns=COLS)
    edges = pd.DataFrame(edge_rows, columns=COLS)
    if len(edges):
        used = set(edges.source_id) | set(edges.target_id)
        nodes = nodes[nodes.node_id.isin(used)].reset_index(drop=True)
    return nodes, edges

# ----------------------------------------------------------------------
# Available timepoints
# ----------------------------------------------------------------------
def avail_T(zp):
    meta = read_array_meta(zp)
    T = meta["shape"][0]
    present = [t for t in range(T) if (Path(zp) / "0" / "c" / str(t) / "0" / "0" / "0").exists()]
    return max(present) + 1 if present else 0

# ----------------------------------------------------------------------
# Run on all test movies
# ----------------------------------------------------------------------
parts = []
for zp in sorted(TEST_DIR.glob("*.zarr")):
    ds = zp.name.replace(".zarr", "")
    T = avail_T(zp)
    if T == 0:
        print("skip", ds)
        continue
    t0 = time.time()
    nodes, edges = track_movie(zp, ds, T)
    parts += [nodes, edges]
    print(f"  {ds}: T={T} nodes={len(nodes)} edges={len(edges)} ({time.time()-t0:.1f}s)")

sub = pd.concat(parts, ignore_index=True)
sub.index.name = "id"
sub.to_csv(OUT)

exp = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
assert list(sub.columns) == exp
print("wrote", OUT, "rows", len(sub), "| nodes", (sub.row_type == 'node').sum(), "edges", (sub.row_type == 'edge').sum())